In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

### **Data Reading**

In [0]:
df_products = spark.sql("select * from databricks_ete_project.bronze.bronze_products")
df_products.display()

In [0]:
df_products.createOrReplaceTempView('products')

### **Functions**

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_ete_project.bronze.discount_func(p_price DOUBLE)
RETURNS DOUBLE
LANGUAGE SQL
RETURN p_price * 0.90

In [0]:
%sql
select product_id, price, databricks_ete_project.bronze.discount_func(price) as discounted_price from products

In [0]:
df_products = df_products.withColumn('discounted_price', expr("databricks_ete_project.bronze.discount_func(price)"))
display(df_products)



In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_ete_project.bronze.upper_func(p_brand STRING)
RETURNS STRING
LANGUAGE PYTHON
AS 
$$
   return p_brand.upper()
$$


In [0]:
%sql
SELECT product_id, brand, databricks_ete_project.bronze.upper_func(brand) as brand_upper
FROM products

In [0]:
df_products.write\
           .format('delta')\
           .mode('overwrite')\
           .option('overwriteSchema', 'true')\
           .saveAsTable('databricks_ete_project.silver.silver_products')

In [0]:
%sql
select * from databricks_ete_project.silver.silver_products